In [129]:
import pandas as pd 

data_tags=pd.read_csv('ml-latest-small/tags.csv')
data_tags.head(20)

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200
5,2,89774,Tom Hardy,1445715205
6,2,106782,drugs,1445715054
7,2,106782,Leonardo DiCaprio,1445715051
8,2,106782,Martin Scorsese,1445715056
9,7,48516,way too long,1169687325


In [130]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf=TfidfVectorizer(stop_words='english')

data_tags['tag']=data_tags['tag'].fillna('')

tfidf_matrix=tfidf.fit_transform(data_tags['tag'])
tfidf_matrix.shape


(3683, 1673)

In [131]:
tfidf.get_feature_names_out()[10:20]

array(['2d', '70mm', '80', 'aardman', 'abortion', 'absorbing', 'abstract',
       'abuse', 'academy', 'accident'], dtype=object)

In [132]:
from sklearn.metrics.pairwise import linear_kernel
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
cosine_sim.shape


(3683, 3683)

In [133]:
cosine_sim[1]

array([0., 1., 0., ..., 0., 0., 0.], shape=(3683,))

In [134]:
data_movies=pd.read_csv('ml-latest-small/movies.csv')
metadata=pd.merge(data_tags,data_movies,on="movieId")
metadata.head(20)

,userId,movieId,tag,timestamp,title,genres
0,2,60756,funny,1445714994,Step Brothers (2008),Comedy
1,2,60756,Highly quotable,1445714996,Step Brothers (2008),Comedy
2,2,60756,will ferrell,1445714992,Step Brothers (2008),Comedy
3,2,89774,Boxing story,1445715207,Warrior (2011),Drama
4,2,89774,MMA,1445715200,Warrior (2011),Drama
5,2,89774,Tom Hardy,1445715205,Warrior (2011),Drama
6,2,106782,drugs,1445715054,"Wolf of Wall Street, The (2013)",Comedy|Crime|Drama
7,2,106782,Leonardo DiCaprio,1445715051,"Wolf of Wall Street, The (2013)",Comedy|Crime|Drama
8,2,106782,Martin Scorsese,1445715056,"Wolf of Wall Street, The (2013)",Comedy|Crime|Drama
9,7,48516,way too long,1169687325,"Departed, The (2006)",Crime|Drama|Thriller


In [135]:
metadata = metadata.drop_duplicates(subset='title').reset_index(drop=True)

In [136]:
indices = pd.Series(metadata.index, index=metadata['title']).drop_duplicates()
indices[:10]

title
Step Brothers (2008)                              0
Warrior (2011)                                    1
Wolf of Wall Street, The (2013)                   2
Departed, The (2006)                              3
Carlito's Way (1993)                              4
Godfather: Part II, The (1974)                    5
Pianist, The (2002)                               6
Lucky Number Slevin (2006)                        7
Fracture (2007)                                   8
Upside Down: The Creation Records Story (2010)    9
dtype: int64

In [137]:
metadata['genres'] = metadata['genres'].str.replace('|', ' ', regex=False).str.lower()

In [138]:
metadata['tag'] = metadata['tag'].fillna('').str.lower()


In [139]:
metadata['soup'] = metadata['tag'].astype(str) + ' ' + metadata['genres'].astype(str)

In [140]:
from sklearn.metrics.pairwise import cosine_similarity
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(metadata['soup'])

In [141]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [142]:
indices = pd.Series(metadata.index, index=metadata['title']).drop_duplicates()

In [143]:
# sažimanje tagova za isti film u jedan dugački tekst
metadata_clean = metadata.groupby('title')['soup'].apply(lambda x: ' '.join(x)).reset_index()
metadata_clean.head(20)


,title,soup
0,(500) Days of Summer (2009),artistic comedy drama romance
1,...And Justice for All (1979),lawyers drama thriller
2,10 Cloverfield Lane (2016),creepy thriller
3,10 Things I Hate About You (1999),shakespeare sort of comedy romance
4,101 Dalmatians (1996),dogs adventure children comedy
5,101 Dalmatians (One Hundred and One Dalmatians...,disney adventure animation children
6,"11'09""01 - September 11 (2002)",terrorism drama
7,12 Angry Men (1957),court drama
8,127 Hours (2010),stranded adventure drama thriller
9,13 Going on 30 (2004),mark ruffalo comedy fantasy romance


In [144]:
test_film = metadata[metadata['title'] == '3:10 to Yuma (2007)']
print(test_film[['userId', 'tag', 'genres']])

     userId             tag                      genres
187     357  christian bale  action crime drama western


In [145]:
import numpy as np

def get_recommendations(title, cosine_sim=cosine_sim):
    idx = indices[title]
    if not isinstance(idx, (int, float, np.integer)):
        idx = idx.iloc[0]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    sim_scores = sim_scores[1:31]
    movie_indices = [i[0] for i in sim_scores]
    results = metadata.iloc[movie_indices]
    
    final_recommendations = results.drop_duplicates('title').head(10)

    return final_recommendations[['title', 'soup']]

In [146]:
get_recommendations('Up (2009)')

,title,soup
21,Toy Story 2 (1999),animation adventure animation children comedy ...
696,101 Dalmatians (One Hundred and One Dalmatians...,disney adventure animation children
1136,Finding Nemo (2003),disney adventure animation children comedy
86,"Croods, The (2013)",animation adventure animation comedy
1477,Watership Down (1978),atmospheric adventure animation children drama...
1457,Sintel (2010),adventure animation fantasy
1366,"Cat Returns, The (Neko no ongaeshi) (2002)",in netflix queue adventure animation children ...
207,Raiders of the Lost Ark (Indiana Jones and the...,adventure action adventure
1346,"Incredibles, The (2004)",disney action adventure animation children comedy
471,Pete's Dragon (1977),disney adventure animation children musical


In [147]:
get_recommendations('Fight Club (1999)')

,title,soup
178,Burn After Reading (2008),dark comedy comedy crime drama
90,Very Bad Things (1998),dark comedy comedy crime
1484,Irreversible (Irréversible) (2002),dark crime drama mystery thriller
115,Memento (2000),dark mystery thriller
1542,Logan (2017),dark action sci-fi
253,Dr. Strangelove or: How I Learned to Stop Worr...,dark comedy comedy war
65,John Wick (2014),dark hero action thriller
98,Funny Games U.S. (2007),dark humor drama thriller
75,John Wick: Chapter Two (2017),action action crime thriller
88,Taken 3 (2015),action action crime thriller


In [148]:
idx_a = indices['Fight Club (1999)']
if not isinstance(idx_a, (int, np.integer)): idx_a = idx_a.iloc[0]

idx_b = indices['Burn After Reading (2008)']
if not isinstance(idx_b, (int, np.integer)): idx_b = idx_b.iloc[0]

similarity = cosine_sim[idx_a][idx_b]
print(f"Sličnost između Fight Club i Burn After Reading  je: {similarity:.4f}")

Sličnost između Fight Club i Burn After Reading  je: 0.8470


In [149]:
def get_soup_info(naslov):
    return metadata[metadata['title'] == naslov]['soup'].iloc[0]

print(f"Sadržaj za Fight Club: {get_soup_info('Fight Club (1999)')}")
print("-" * 50)
print(f"Sadržaj za Natural Born Killers: {get_soup_info('Burn After Reading (2008)')}")

Sadržaj za Fight Club: dark comedy action crime drama thriller
--------------------------------------------------
Sadržaj za Natural Born Killers: dark comedy comedy crime drama


In [150]:
get_recommendations('Star Wars: Episode III - Revenge of the Sith (2005)')

,title,soup
650,Armageddon (1998),space action romance sci-fi thriller
1265,Babylon 5: In the Beginning (1998),space station adventure sci-fi
114,X-Men (2000),action action adventure sci-fi
72,Batman v Superman: Dawn of Justice (2016),action action adventure fantasy sci-fi
208,Aliens (1986),action action adventure horror sci-fi
199,Blade Runner (1982),sci-fi action sci-fi thriller
210,"Terminator, The (1984)",action action sci-fi thriller
1560,Final Fantasy: The Spirits Within (2001),sci-fi adventure animation fantasy sci-fi
64,"Maze Runner, The (2014)",action action mystery sci-fi
772,Superman (1978),superhero action adventure sci-fi


In [151]:
idx_a = indices['Star Wars: Episode III - Revenge of the Sith (2005)']
if not isinstance(idx_a, (int, np.integer)): idx_a = idx_a.iloc[0]

idx_b = indices['Star Wars: Episode VI - Return of the Jedi (1983)']
if not isinstance(idx_b, (int, np.integer)): idx_b = idx_b.iloc[0]

similarity = cosine_sim[idx_a][idx_b]
print(f"Sličnost između prva dva gore filma je: {similarity:.4f}")

Sličnost između prva dva gore filma je: 0.3310


In [152]:
import re

# Funkcija za čišćenje naslova (da ostane samo tekst, bez godine)
def clean_title(title):
    return re.sub(r'\s*\(\d{4}\)', '', title).lower()

In [153]:
metadata['clean_title'] = metadata['title'].apply(clean_title)
metadata.head(10)

,userId,movieId,tag,timestamp,title,genres,soup,clean_title
0,2,60756,funny,1445714994,Step Brothers (2008),comedy,funny comedy,step brothers
1,2,89774,boxing story,1445715207,Warrior (2011),drama,boxing story drama,warrior
2,2,106782,drugs,1445715054,"Wolf of Wall Street, The (2013)",comedy crime drama,drugs comedy crime drama,"wolf of wall street, the"
3,7,48516,way too long,1169687325,"Departed, The (2006)",crime drama thriller,way too long crime drama thriller,"departed, the"
4,18,431,al pacino,1462138765,Carlito's Way (1993),crime drama,al pacino crime drama,carlito's way
5,18,1221,al pacino,1461699306,"Godfather: Part II, The (1974)",crime drama,al pacino crime drama,"godfather: part ii, the"
6,18,5995,holocaust,1455735472,"Pianist, The (2002)",drama war,holocaust drama war,"pianist, the"
7,18,44665,twist ending,1456948283,Lucky Number Slevin (2006),crime drama mystery,twist ending crime drama mystery,lucky number slevin
8,18,52604,anthony hopkins,1457650696,Fracture (2007),crime drama mystery thriller,anthony hopkins crime drama mystery thriller,fracture
9,18,88094,britpop,1457444500,Upside Down: The Creation Records Story (2010),documentary,britpop documentary,upside down: the creation records story


In [154]:
metadata['title_soup'] = (metadata['clean_title'] + " ") * 2 + metadata['tag'].astype(str) + ' ' + metadata['genres'].astype(str)
metadata

,userId,movieId,tag,timestamp,title,genres,soup,clean_title,title_soup
0,2,60756,funny,1445714994,Step Brothers (2008),comedy,funny comedy,step brothers,step brothers step brothers funny comedy
1,2,89774,boxing story,1445715207,Warrior (2011),drama,boxing story drama,warrior,warrior warrior boxing story drama
2,2,106782,drugs,1445715054,"Wolf of Wall Street, The (2013)",comedy crime drama,drugs comedy crime drama,"wolf of wall street, the","wolf of wall street, the wolf of wall street, ..."
3,7,48516,way too long,1169687325,"Departed, The (2006)",crime drama thriller,way too long crime drama thriller,"departed, the","departed, the departed, the way too long crime..."
4,18,431,al pacino,1462138765,Carlito's Way (1993),crime drama,al pacino crime drama,carlito's way,carlito's way carlito's way al pacino crime drama
...,...,...,...,...,...,...,...,...,...
1567,606,1948,british,1177512649,Tom Jones (1963),adventure comedy romance,british adventure comedy romance,tom jones,tom jones tom jones british adventure comedy r...
1568,606,5694,70mm,1175638092,Staying Alive (1983),comedy drama musical,70mm comedy drama musical,staying alive,staying alive staying alive 70mm comedy drama ...
1569,606,6107,world war ii,1178473747,Night of the Shooting Stars (Notte di San Lore...,drama war,world war ii drama war,night of the shooting stars (notte di san lore...,night of the shooting stars (notte di san lore...
1570,606,7936,austere,1173392334,Shame (Skammen) (1968),drama war,austere drama war,shame (skammen),shame (skammen) shame (skammen) austere drama war


In [155]:
from sklearn.metrics.pairwise import cosine_similarity
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(metadata['title_soup'])

In [156]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [157]:
def get_recommendations_with_title(title, cosine_sim=cosine_sim):
    # 1. Uzmi prvu riječ iz unosa (npr. 'Batman' ako uneseš 'Batman Forever')
    # Ili pretraži po cijelom pojmu ali bez brige o zagradama
    search_term = title.split('(')[0].strip() # Ovo miče (1995) iz pretrage
    
    try:
        # Tražimo filmove koji sadrže taj očišćeni naslov
        # regex=False sprečava probleme sa zagradama
        matches = metadata[metadata['title'].str.contains(search_term, case=False, na=False, regex=False)]
        
        if matches.empty:
            return "Film nije pronađen."
            
        idx = matches.index[0] # Uzimamo prvi pronađeni film
    except Exception as e:
        return f"Greška: {e}"

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    sim_scores = sim_scores[1:11]
    movie_indices = [i[0] for i in sim_scores]

    return metadata[['title', 'genres']].iloc[movie_indices]

In [158]:
get_recommendations_with_title('Batman Forever (1995)')

,title,genres
375,Batman (1989),action crime thriller
587,Batman Returns (1992),action crime
1541,The Lego Batman Movie (2017),action animation comedy
72,Batman v Superman: Dawn of Justice (2016),action adventure fantasy sci-fi
772,Superman (1978),action adventure sci-fi
787,Mystery Men (1999),action comedy fantasy
919,Supergirl (1984),action adventure fantasy
1033,Spider-Man (2002),action adventure sci-fi thriller
774,Superman III (1983),action adventure sci-fi
773,Superman II (1980),action sci-fi


In [159]:
idx_a = indices['Batman Forever (1995)']
if not isinstance(idx_a, (int, np.integer)): idx_a = idx_a.iloc[0]

idx_b = indices['Batman (1989)']
if not isinstance(idx_b, (int, np.integer)): idx_b = idx_b.iloc[0]

similarity = cosine_sim[idx_a][idx_b]
print(f"Sličnost između prva dva gore filma je: {similarity:.4f}")

Sličnost između prva dva gore filma je: 0.6793


In [160]:
get_recommendations('Batman Forever (1995)')

,title,soup
587,Batman Returns (1992),superhero action crime
375,Batman (1989),superhero action crime thriller
919,Supergirl (1984),superhero action adventure fantasy
787,Mystery Men (1999),superhero action comedy fantasy
772,Superman (1978),superhero action adventure sci-fi
774,Superman III (1983),superhero action adventure sci-fi
1033,Spider-Man (2002),superhero action adventure sci-fi thriller
773,Superman II (1980),superhero action sci-fi
927,Unbreakable (2000),superhero drama sci-fi
51,Pineapple Express (2008),comedy action comedy crime


In [161]:
get_recommendations('Star Wars: Episode IV - A New Hope (1977)')

,title,soup
1556,"War of the Worlds, The (1953)",classic action drama sci-fi
114,X-Men (2000),action action adventure sci-fi
72,Batman v Superman: Dawn of Justice (2016),action action adventure fantasy sci-fi
208,Aliens (1986),action action adventure horror sci-fi
199,Blade Runner (1982),sci-fi action sci-fi thriller
210,"Terminator, The (1984)",action action sci-fi thriller
1560,Final Fantasy: The Spirits Within (2001),sci-fi adventure animation fantasy sci-fi
64,"Maze Runner, The (2014)",action action mystery sci-fi
772,Superman (1978),superhero action adventure sci-fi
774,Superman III (1983),superhero action adventure sci-fi


In [169]:
idx_a = indices['Star Wars: Episode IV - A New Hope (1977)']
if not isinstance(idx_a, (int, np.integer)): idx_a = idx_a.iloc[0]

idx_b = indices['War of the Worlds, The (1953)']
if not isinstance(idx_b, (int, np.integer)): idx_b = idx_b.iloc[0]

similarity = cosine_sim[idx_a][idx_b]
print(f"Sličnost između prva dva gore filma je: {similarity:.4f}")

Sličnost između prva dva gore filma je: 0.1288


In [163]:
get_recommendations_with_title('Star Wars: Episode IV - A New Hope (1977)')

,title,genres
506,Star Wars: Episode V - The Empire Strikes Back...,action adventure sci-fi
771,Star Wars: Episode I - The Phantom Menace (1999),action adventure sci-fi
1391,Star Wars: Episode III - Revenge of the Sith (...,action adventure sci-fi
515,Star Wars: Episode VI - Return of the Jedi (1983),action adventure sci-fi
81,Solo: A Star Wars Story (2018),action adventure children sci-fi
586,Star Trek IV: The Voyage Home (1986),adventure comedy sci-fi
940,Hope and Glory (1987),drama
742,Rocky IV (1985),action drama
1445,Star Trek (2009),action adventure sci-fi imax
578,Star Trek: First Contact (1996),action adventure sci-fi thriller


In [166]:
idx_a = indices['Star Wars: Episode IV - A New Hope (1977)']
if not isinstance(idx_a, (int, np.integer)): idx_a = idx_a.iloc[0]

idx_b = indices['Star Wars: Episode V - The Empire Strikes Back (1980)']
if not isinstance(idx_b, (int, np.integer)): idx_b = idx_b.iloc[0]

similarity = cosine_sim[idx_a][idx_b]
print(f"Sličnost između prva dva gore filma je: {similarity:.4f}")

Sličnost između prva dva gore filma je: 0.4824


In [167]:
idx_a = indices['Star Wars: Episode IV - A New Hope (1977)']
if not isinstance(idx_a, (int, np.integer)): idx_a = idx_a.iloc[0]

idx_b = indices['Hope and Glory (1987)']
if not isinstance(idx_b, (int, np.integer)): idx_b = idx_b.iloc[0]

similarity = cosine_sim[idx_a][idx_b]
print(f"Sličnost između prva dva gore filma je: {similarity:.4f}")

Sličnost između prva dva gore filma je: 0.2900
